In [9]:
import os, json
import librosa
import torch
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    AutoTokenizer,
    AutoModelForCausalLM,
    Wav2Vec2Processor, Wav2Vec2Model, Wav2Vec2FeatureExtractor
)
import torch.nn.functional as F

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

TEST_DIR = "dataset/test_public"        # grade-time: hidden eval set is overlaid here
OUTPUT   = "answers.json"

device = "cuda" if torch.cuda.is_available() else "cpu"

with open(os.path.join(TEST_DIR, "prefix.json")) as f:
    prefixes = json.load(f)

whisper_processor = WhisperProcessor.from_pretrained(
    "models/whisper-small",
    local_files_only=True,
)

whisper_model = WhisperForConditionalGeneration.from_pretrained(
    "models/whisper-small",
    dtype=torch.float16 if device == "cuda" else torch.float32,
    local_files_only=True,
).to(device).eval()

tokenizer = AutoTokenizer.from_pretrained(
    "models/qwen2.5-0.5b",
    local_files_only=True,
)

llm_model = AutoModelForCausalLM.from_pretrained(
    "models/qwen2.5-0.5b",
    dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    local_files_only=True,
).to(device).eval()

dialogue_ids = sorted(
    (d for d in os.listdir(TEST_DIR) if os.path.isdir(os.path.join(TEST_DIR, d))),
    key=lambda x: int(x) if x.isdigit() else x,
)

WAV2VEC_PATH = "models/wav2vec2-base-960h"
wav2vec_processor = Wav2Vec2Processor.from_pretrained(WAV2VEC_PATH)

wav2vec_model = Wav2Vec2Model.from_pretrained(WAV2VEC_PATH, local_files_only=True).to(device).eval()

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(
    WAV2VEC_PATH,
    local_files_only=True
)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: models/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.bias      | UNEXPECTED | 
lm_head.weight    | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [17]:
def score_next_candidate(context, candidate):
    prompt = ("The following is a conversation between two people:\n" + "\n".join(context) + "\nNext turn:")

    context_ids = tokenizer.encode(prompt, add_special_tokens=False)
    candidate_ids = tokenizer.encode(" " + candidate, add_special_tokens=False)

    if not candidate_ids:
        return float("-inf")

    input_ids = torch.tensor(
        [context_ids + candidate_ids],
        dtype=torch.long,
        device=device
    )

    start = len(context_ids)

    with torch.inference_mode():
        outputs = llm_model(input_ids=input_ids)

        logits = outputs.logits[0, start - 1:-1, :]
        targets = input_ids[0, start:]

        log_probs = F.log_softmax(logits.float(), dim=-1)

        token_scores = log_probs.gather(
            dim=-1,
            index=targets.unsqueeze(-1)
        ).squeeze(-1)

        return token_scores.mean().item()

In [18]:
def reconstruct_order(transcripts, first, second, temperature=1.0):
    if temperature <= 0:
        raise ValueError("temperature must be positive")

    texts = {
        item["id"]: item["text"]
        for item in transcripts
    }

    sequence = [first, second]
    remaining = sorted(
        chunk_id for chunk_id in texts
        if chunk_id not in sequence
    )

    history = []

    while remaining:
        context = [texts[chunk_id] for chunk_id in sequence]

        scores = [
            score_next_candidate(context, texts[chunk_id])
            for chunk_id in remaining
        ]

        score_tensor = torch.tensor(scores, dtype=torch.float32)

        if not torch.isfinite(score_tensor).any():
            raise ValueError("No candidate has a finite score")

        probabilities = torch.softmax(
            score_tensor / temperature, dim=0
        ).tolist()

        ranking = [
            {
                "id": chunk_id,
                "text": texts[chunk_id],
                "score": score,
                "probability": probability,
            }
            for chunk_id, score, probability in zip(
                remaining, scores, probabilities
            )
        ]

        ranking.sort(
            key=lambda item: item["score"],
            reverse=True,
        )

        best_id = ranking[0]["id"]

        history.append({
            "context_ids": sequence.copy(),
            "candidates": ranking,
            "selected_id": best_id,
        })

        sequence.append(best_id)
        remaining.remove(best_id)

    return sequence, history

In [16]:
context = [
    "A: What time does the meeting start?",
    "B: It starts at three o'clock."
]

candidates = [
    "A: Where is the meeting?",
    "A: Did you have breakfast?",
    "A: Thank you, see you there!"
]

scores = [
    score_next_candidate(context, candidate)
    for candidate in candidates
]

for candidate, score in zip(candidates, scores):
    print(f"{score:.4f} | {candidate}")

-1.5249 | A: Where is the meeting?
-2.1425 | A: Did you have breakfast?
-2.8781 | A: Thank you, see you there!


In [12]:
def speaker_embedding(audio_path):
    audio, _ = librosa.load(
        str(audio_path),
        sr=16000,
        duration=12
    )

    inputs = feature_extractor(
        audio,
        sampling_rate=16000,
        return_tensors="pt"
    )

    with torch.inference_mode():
        outputs = wav2vec_model(
            inputs.input_values.to(device),
            output_hidden_states=True
        )

    hidden = outputs.hidden_states[6][0]

    embedding = hidden.mean(dim=0)

    embedding = F.normalize(
        embedding.float(),
        dim=0
    )

    return embedding.cpu().numpy()

In [14]:
from pathlib import Path
import numpy as np

a = speaker_embedding(
    Path("dataset/train/0/chunk_0.wav")
)

b = speaker_embedding(
    Path("dataset/train/0/chunk_1.wav")
)

similarity = np.dot(a, b)

print("Similarity:", similarity)

Similarity: 0.65990317


In [19]:
def trans_audio(audio_path):
    au, _ = librosa.load(audio_path, sr=16000)

    features = whisper_processor(au, sampling_rate=16000, return_tensors="pt",).input_features.to(device=device, dtype=next(whisper_model.parameters()).dtype)

    with torch.inference_mode():
        ids = whisper_model.generate(features, language="en", task="transcribe", max_new_tokens=128)

    return whisper_processor.batch_decode(ids, skip_special_tokens=True)[0].strip()

In [22]:
import numpy as np
from sklearn.cluster import AgglomerativeClustering

def identify_speakers(files, first):
    chunk_ids = [
        int(file.stem.split("_")[1])
        for file in files
    ]

    embeddings = np.stack([
        speaker_embedding(file)
        for file in files
    ])

    labels = AgglomerativeClustering(
        n_clusters=2,
        metric="cosine",
        linkage="average",
    ).fit_predict(embeddings)

    first_label = labels[chunk_ids.index(first)]

    speakers = {
        chunk_id: int(label != first_label)
        for chunk_id, label in zip(chunk_ids, labels)
    }

    return speakers

In [27]:
import json
from pathlib import Path

SPLIT = "test_leaderboard_a"
TEST_DIR = f"hf-data/private/{SPLIT}"
OUTPUT = f"answers_{SPLIT}.json"

prefixes = json.loads(
    (Path(TEST_DIR) / "prefix.json").read_text()
)

assert len(prefixes) == 100
assert all((Path(TEST_DIR) / did).is_dir() for did in prefixes)

print("Набор:", SPLIT)
print("Диалогов:", len(prefixes))
print("Файл для предсказаний:", OUTPUT)

Набор: test_leaderboard_a
Диалогов: 100
Файл для предсказаний: answers_test_leaderboard_a.json


In [28]:
# Final
import os, json
import time
from pathlib import Path

answers = {}
experiment_details = {}

start_time = time.perf_counter()

for did, (first, second) in prefixes.items():
    folder = Path(TEST_DIR) / did

    files = sorted(folder.glob("chunk_*.wav"), key=lambda p: int(p.stem.split("_")[1]),)

    n = len(files)
    transcripts = []

    for file in files:
        chunk_id = int(file.stem.split("_")[1])

        text = trans_audio(str(file))

        transcripts.append({
            "id": chunk_id,
            "text": text,
        })

    sequence, history = reconstruct_order(transcripts, first, second, temperature=1)

    order = [0] * n

    for pos, chunk_id in enumerate(sequence):
        order[chunk_id] = pos

    answers[did] = order

    experiment_details[did] = {
        "trans": transcripts,
        "sequence": sequence,
        "history": history,
    }

    print(f"\nDialogue {did}: {sequence}")

    for step in history:
        best = step["candidates"][0]

        print(
            f'  Context {step["context_ids"]} '
            f'→ chunk {best["id"]} '
            f'| weight={best["probability"]:.1%} '
            f'| {best["text"]}'
        )

with open(OUTPUT, "w") as f:
    json.dump(answers, f)

with open("qwen_scoring_details.json", "w") as f:
    json.dump(
        experiment_details,
        f,
        ensure_ascii=False,
        indent=2,
    )

print("\nFinished!")
print(f"Dialogues processed: {len(answers)}")
print(f"Answers: {OUTPUT}")
print("Details: qwen_scoring_details.json")
print(f"Elapsed: {time.perf_counter() - start_time:.1f}s")

[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface


Dialogue 691: [3, 0, 6, 1, 5, 4, 2]
  Context [3, 0] → chunk 6 | weight=53.6% | Sorry, I don't think I can.
  Context [3, 0, 6] → chunk 1 | weight=84.7% | Then you've got to try it! Let's go to the skating rink on Thursday!
  Context [3, 0, 6, 1] → chunk 5 | weight=57.9% | Friday then, how about it?
  Context [3, 0, 6, 1, 5] → chunk 4 | weight=88.5% | Friday is great!
  Context [3, 0, 6, 1, 5, 4] → chunk 2 | weight=100.0% | Is Settled in?


[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface


Dialogue 895: [0, 1, 5, 4, 3, 2, 6]
  Context [0, 1] → chunk 5 | weight=34.5% | Yeah, well, I'll see you this weekend.
  Context [0, 1, 5] → chunk 4 | weight=49.7% | Bummer! I guess we're both out of luck.
  Context [0, 1, 5, 4] → chunk 3 | weight=74.5% | It's not worth it. Besides, I have to save up to pay for some bills.
  Context [0, 1, 5, 4, 3] → chunk 2 | weight=54.1% | But you've been looking for destined for years.
  Context [0, 1, 5, 4, 3, 2] → chunk 6 | weight=100.0% | and let them ruin my life? Forget it.


[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface


Dialogue 305: [2, 4, 5, 1, 6, 3, 0]
  Context [2, 4] → chunk 5 | weight=31.5% | Do you mind if I read your newspaper for a while?
  Context [2, 4, 5] → chunk 1 | weight=36.9% | Sorry, I don't know. I'm a passenger waiting for my train.
  Context [2, 4, 5, 1] → chunk 6 | weight=40.7% | Do you know if Train 10 is on time to be here?
  Context [2, 4, 5, 1, 6] → chunk 3 | weight=65.6% | Thanks a lot.
  Context [2, 4, 5, 1, 6, 3] → chunk 0 | weight=100.0% | Certainly not. Here it is.


... (190 outputs left out)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface


Dialogue 940: [5, 10, 11, 9, 7, 1, 19, 0, 13, 14, 3, 6, 17, 16, 12, 15, 8, 4, 2, 18]
  Context [5, 10] → chunk 11 | weight=14.7% | How long will you be staying?
  Context [5, 10, 11] → chunk 9 | weight=14.6% | Yes, it is. Thank you.
  Context [5, 10, 11, 9] → chunk 7 | weight=13.4% | What kind of room would you like, madam?
  Context [5, 10, 11, 9, 7] → chunk 1 | weight=13.3% | That's fine.
  Context [5, 10, 11, 9, 7, 1] → chunk 19 | weight=15.2% | Certainly, madam. I'll just check what we have available. Yes, we have a room on the fourth floor with a really splendid view.
  Context [5, 10, 11, 9, 7, 1, 19] → chunk 0 | weight=19.6% | Okay, let me make sure I got that.
  Context [5, 10, 11, 9, 7, 1, 19, 0] → chunk 13 | weight=20.7% | Let me give you your confirmation number. It's 757-6385. Thank you for choosing San Feliz Hotel and have a nice day. Goodbye.
  Context [5, 10, 11, 9, 7, 1, 19, 0, 13] → chunk 14 | weight=20.3% | March 23rd.
  Context [5, 10, 11, 9, 7, 1, 19, 0, 13, 14] → 

[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface


Dialogue 501: [14, 5, 10, 0, 11, 17, 1, 4, 12, 8, 16, 3, 15, 19, 6, 18, 2, 9, 7, 13]
  Context [14, 5] → chunk 10 | weight=17.2% | I can help you with that.
  Context [14, 5, 10] → chunk 0 | weight=18.2% | Did you have anything specific in mind?
  Context [14, 5, 10, 0] → chunk 11 | weight=10.3% | Well, you made a good choice. Coming to us, I mean, we have a full range of products from cosmetics to skin cleansers and moisturizers.
  Context [14, 5, 10, 0, 11] → chunk 17 | weight=10.1% | We have a very wide selection of colors and our eye shadow is specially designed to moisturize the skin. Would you like to sample some?
  Context [14, 5, 10, 0, 11, 17] → chunk 1 | weight=12.9% | Let me show you. We have this rose color. We also have different shades of beige.
  Context [14, 5, 10, 0, 11, 17, 1] → chunk 4 | weight=11.8% | Of course, here is our thickening mascara. We also have mascara designed to lengthen the eyelashes.
  Context [14, 5, 10, 0, 11, 17, 1, 4] → chunk 12 | weight=15.6% |

In [23]:
import json
import random
import time
from pathlib import Path

EVAL_DIR = Path("dataset/train")
TRUTH_PATH = Path("hf-data/public/train_answers.json")

N_DIALOGUES = 100
SEED = 42

with open(EVAL_DIR / "prefix.json") as f:
    eval_prefixes = json.load(f)

with open(TRUTH_PATH) as f:
    truth = json.load(f)

available_ids = sorted(eval_prefixes, key=int)

assert len(available_ids) >= N_DIALOGUES
assert set(available_ids) <= set(truth)

selected_ids = random.Random(SEED).sample(
    available_ids,
    N_DIALOGUES,
)

answers_by_variant = {
    "text_only": {},
    "with_speakers": {},
}

experiment_details = {}

start_time = time.perf_counter()

for number, did in enumerate(selected_ids, start=1):
    first, second = eval_prefixes[did]
    folder = EVAL_DIR / did

    files = sorted(
        folder.glob("chunk_*.wav"),
        key=lambda p: int(p.stem.split("_")[1]),
    )

    chunk_ids = [
        int(file.stem.split("_")[1])
        for file in files
    ]

    n = len(files)

    assert chunk_ids == list(range(n)), did
    assert n == len(truth[did]), did

    transcripts = []

    for file, chunk_id in zip(files, chunk_ids):
        transcripts.append({
            "id": chunk_id,
            "text": trans_audio(str(file)),
        })

    speakers = identify_speakers(files, first)

    speaker_transcripts = [
        {
            "id": item["id"],
            "text": (
                f'Speaker {speakers[item["id"]]}: '
                f'{item["text"]}'
            ),
        }
        for item in transcripts
    ]

    variants = {
        "text_only": transcripts,
        "with_speakers": speaker_transcripts,
    }

    details = {
        "transcripts": transcripts,
        "speakers": speakers,
        "variants": {},
    }

    for name, variant_transcripts in variants.items():
        sequence, history = reconstruct_order(
            variant_transcripts,
            first,
            second,
            temperature=1.0,
        )

        assert sorted(sequence) == chunk_ids
        assert sequence[:2] == [first, second]

        order = [0] * n

        for pos, chunk_id in enumerate(sequence):
            order[chunk_id] = pos

        answers_by_variant[name][did] = order

        details["variants"][name] = {
            "sequence": sequence,
            "history": history,
        }

    experiment_details[did] = details

    for name, predictions in answers_by_variant.items():
        with open(f"answers_train100_{name}.json", "w") as f:
            json.dump(predictions, f)

    with open("details_train100.json", "w") as f:
        json.dump(
            {
                "seed": SEED,
                "selected_ids": selected_ids,
                "dialogues": experiment_details,
            },
            f,
            ensure_ascii=False,
            indent=2,
        )

    print(
        f"[{number}/{N_DIALOGUES}] Dialogue {did} completed",
        flush=True,
    )

print(f"Elapsed: {time.perf_counter() - start_time:.1f}s")

[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

[1/100] Dialogue 441 completed


[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

[2/100] Dialogue 89 completed


[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

[3/100] Dialogue 1096 completed


... (190 outputs left out)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

[99/100] Dialogue 1597 completed


[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

[100/100] Dialogue 1438 completed
Elapsed: 239.6s


In [24]:
def score_dialogue(prediction, target):
    n = len(target)

    if len(prediction) != n:
        return 0.0

    if sorted(prediction) != list(range(n)):
        return 0.0

    if n < 2:
        return 1.0

    correct = sum(
        (prediction[i] - prediction[j])
        * (target[i] - target[j]) > 0
        for i in range(n)
        for j in range(i + 1, n)
    )

    return correct / (n * (n - 1) // 2)


baseline_answers = {}

for did in selected_ids:
    first, second = eval_prefixes[did]
    n = len(truth[did])

    sequence = [first, second] + [
        i for i in range(n)
        if i not in (first, second)
    ]

    order = [0] * n

    for pos, chunk_id in enumerate(sequence):
        order[chunk_id] = pos

    baseline_answers[did] = order


all_predictions = {
    "baseline": baseline_answers,
    **answers_by_variant,
}

scores_by_variant = {}

for name, predictions in all_predictions.items():
    assert set(predictions) == set(selected_ids)

    scores = [
        score_dialogue(predictions[did], truth[did])
        for did in selected_ids
    ]

    scores_by_variant[name] = np.array(scores)

    print(f"{name:>15}: {np.mean(scores):.6f}")


difference = (
    scores_by_variant["with_speakers"]
    - scores_by_variant["text_only"]
)

print()
print(f"Mean speaker improvement: {difference.mean():+.6f}")
print(f"Improved: {np.sum(difference > 1e-9)}")
print(f"Worse:    {np.sum(difference < -1e-9)}")
print(f"Equal:    {np.sum(np.abs(difference) <= 1e-9)}")

       baseline: 0.675501
      text_only: 0.725800
  with_speakers: 0.709472

Mean speaker improvement: -0.016328
Improved: 38
Worse:    51
Equal:    11


In [25]:
# Scoring
import json
from pathlib import Path

pred = json.loads(Path("answers.json").read_text())
truth = json.loads(
    Path("hf-data/public/pretest_answers.json").read_text()
)

def score_dialogue(p, t):
    n = len(t)

    if len(p) != n or sorted(p) != list(range(n)):
        return 0.0

    if n < 2:
        return 1.0

    correct = sum(
        (p[i] - p[j]) * (t[i] - t[j]) > 0
        for i in range(n)
        for j in range(i + 1, n)
    )

    return correct / (n * (n - 1) // 2)

scores = [
    score_dialogue(pred[k], truth[k])
    for k in truth
]

print(f"IOAI Score: {sum(scores) / len(scores):.6f}")
print(f"Dialogues evaluated: {len(scores)}")

IOAI Score: 0.704561
Dialogues evaluated: 20


In [29]:
SPLIT = "test_leaderboard_a"

TEST_DIR = f"hf-data/private/{SPLIT}"
OUTPUT = f"answers_{SPLIT}.json"

predictions = json.loads(Path(OUTPUT).read_text())
targets = json.loads(
    Path(f"hf-data/private/{SPLIT}_answers.json").read_text()
)

def valid_permutation(p, n):
    return (
        isinstance(p, list)
        and len(p) == n
        and all(type(x) is int for x in p)
        and sorted(p) == list(range(n))
    )

def score_one(prediction, target):
    n = len(target)
    if n < 2:
        return 1.0
    if not valid_permutation(prediction, n):
        return 0.0

    wrong = sum(
        (target[i] > target[j]) != (prediction[i] > prediction[j])
        for i in range(n)
        for j in range(i + 1, n)
    )
    return 1.0 - wrong / (n * (n - 1) // 2)

scores = [
    score_one(predictions.get(did), target)
    for did, target in targets.items()
]

valid = sum(
    valid_permutation(predictions.get(did), len(target))
    for did, target in targets.items()
)

raw = sum(scores) / len(scores)

print("Split:", SPLIT)
print(f"Correct: {valid}/{len(targets)}")
print(f"Pairwise accuracy: {raw:.8f}")
print(f"Points IOAI: {round(100 * raw, 4):.4f}")
print("Unused ID:", sorted(set(predictions) - set(targets)))

Split: test_leaderboard_a
Correct: 100/100
Pairwise accuracy: 0.71248690
Points IOAI: 71.2487
Unused ID: []
